In [2]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import time
import pandas as pd

from models.transfer_models import TransferModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [3]:
data_dir = '../dataSet/archive/seg_train/seg_train'
test_dir = '../dataSet/archive/seg_test/seg_test'

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_train = datasets.ImageFolder(data_dir, transform=transform)
train_size = int(0.8 * len(full_train))
val_size = len(full_train) - train_size
train_dataset, val_dataset = random_split(full_train, [train_size, val_size])
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

Train: 11227, Val: 2807, Test: 3000


In [4]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [5]:
def train_model(model_name, mode='frozen', num_epochs=20, lr=0.001,
                lr_layer4=0.0001, lr_head=0.001, experiment_name=''):
    print(f'Experiment: {experiment_name}')
    print(f'Model: {model_name}, Mode: {mode}')

    model = TransferModel(model_name=model_name, num_classes=6, pretrained=True).to(device)

    if mode == 'frozen':
        model.freeze_all_except_head()
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    elif mode == 'finetuned':
        if model_name in ['resnet18', 'resnet50']:
            model.unfreeze_last_layers_resnet()
            param_groups = model.get_resnet_param_groups(lr_layer4=lr_layer4, lr_head=lr_head)
            optimizer = optim.Adam(param_groups)
        else:
            model.freeze_all_except_head()
            optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable_params:,} / {total_params:,}')

    criterion = nn.CrossEntropyLoss()
    early_stopping = EarlyStopping(patience=5)

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    best_val_acc = 0.0
    start_time = time.time()

    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'Epoch [{epoch+1}/{num_epochs}] Train: {train_acc:.4f}, Val: {val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'../checkpoints/{experiment_name}.pth')

        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f'Early stopping at epoch {epoch+1}')
            break

    training_time = time.time() - start_time

    model.load_state_dict(torch.load(f'../checkpoints/{experiment_name}.pth'))
    test_loss, test_acc = validate(model, test_loader, criterion, device)


    results = {
        'experiment': experiment_name,
        'model': model_name,
        'mode': mode,
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'training_time': training_time,
        'history': history
    }

    if mode == 'finetuned' and model_name in ['resnet18', 'resnet50']:
        results['lr_layer4'] = lr_layer4
        results['lr_head'] = lr_head
    else:
        results['lr'] = lr

    print(f'\nBest Val: {best_val_acc:.4f}, Test: {test_acc:.4f}, Time: {training_time/60:.2f} min')

    return results

In [6]:
results_list = []
result1 = train_model(
    model_name='resnet18',
    mode='frozen',
    num_epochs=15,
    lr=0.001,
    experiment_name='resnet18_frozen'
)
results_list.append(result1)

Experiment: resnet18_frozen
Model: resnet18, Mode: frozen


C:\Users\zhenu\anaconda3\envs\torch_gpu\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\zhenu\anaconda3\envs\torch_gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Trainable: 3,078 / 11,179,590
Epoch [1/15] Train: 0.8216, Val: 0.8995
Epoch [2/15] Train: 0.8915, Val: 0.9152
Epoch [3/15] Train: 0.8948, Val: 0.9223
Epoch [4/15] Train: 0.8965, Val: 0.9156
Epoch [5/15] Train: 0.9048, Val: 0.9141
Epoch [6/15] Train: 0.9043, Val: 0.9202
Epoch [7/15] Train: 0.9078, Val: 0.9181
Epoch [8/15] Train: 0.9079, Val: 0.9156
Epoch [9/15] Train: 0.9091, Val: 0.9216
Epoch [10/15] Train: 0.9103, Val: 0.9184
Epoch [11/15] Train: 0.9122, Val: 0.9088
Epoch [12/15] Train: 0.9113, Val: 0.9113
Epoch [13/15] Train: 0.9137, Val: 0.9191
Epoch [14/15] Train: 0.9116, Val: 0.9202
Epoch [15/15] Train: 0.9115, Val: 0.9074
Early stopping at epoch 15


C:\Users\zhenu\AppData\Local\Temp\ipykernel_31940\2329935758.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'../checkpoints/{experime


Best Val: 0.9223, Test: 0.9063, Time: 9.37 min


In [7]:
result2 = train_model(
    model_name='resnet18',
    mode='finetuned',
    num_epochs=15,
    lr_layer4=0.0001,
    lr_head=0.001,
    experiment_name='resnet18_finetuned'
)
results_list.append(result2)

Experiment: resnet18_finetuned
Model: resnet18, Mode: finetuned
Trainable: 8,396,806 / 11,179,590
Epoch [1/15] Train: 0.8961, Val: 0.9398
Epoch [2/15] Train: 0.9581, Val: 0.9412
Epoch [3/15] Train: 0.9823, Val: 0.9366
Epoch [4/15] Train: 0.9891, Val: 0.9330
Epoch [5/15] Train: 0.9921, Val: 0.9380
Epoch [6/15] Train: 0.9924, Val: 0.9270
Epoch [7/15] Train: 0.9913, Val: 0.9280
Early stopping at epoch 7


C:\Users\zhenu\AppData\Local\Temp\ipykernel_31940\2329935758.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'../checkpoints/{experime


Best Val: 0.9412, Test: 0.9333, Time: 4.35 min


In [8]:
result3 = train_model(
    model_name='resnet50',
    mode='frozen',
    num_epochs=15,
    lr=0.001,
    experiment_name='resnet50_frozen'
)
results_list.append(result3)

Experiment: resnet50_frozen
Model: resnet50, Mode: frozen


C:\Users\zhenu\anaconda3\envs\torch_gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Trainable: 12,294 / 23,520,326
Epoch [1/15] Train: 0.8486, Val: 0.8995
Epoch [2/15] Train: 0.8891, Val: 0.9002
Epoch [3/15] Train: 0.8937, Val: 0.9198
Epoch [4/15] Train: 0.9025, Val: 0.9081
Epoch [5/15] Train: 0.9011, Val: 0.9099
Epoch [6/15] Train: 0.9052, Val: 0.9059
Epoch [7/15] Train: 0.9041, Val: 0.9159
Epoch [8/15] Train: 0.9100, Val: 0.9191
Epoch [9/15] Train: 0.9107, Val: 0.9020
Epoch [10/15] Train: 0.9117, Val: 0.9227
Epoch [11/15] Train: 0.9113, Val: 0.8753
Epoch [12/15] Train: 0.9163, Val: 0.9191
Epoch [13/15] Train: 0.9181, Val: 0.8913
Epoch [14/15] Train: 0.9202, Val: 0.9245
Epoch [15/15] Train: 0.9217, Val: 0.9099


C:\Users\zhenu\AppData\Local\Temp\ipykernel_31940\2329935758.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'../checkpoints/{experime


Best Val: 0.9245, Test: 0.9120, Time: 12.84 min


In [9]:
result4 = train_model(
    model_name='resnet50',
    mode='finetuned',
    num_epochs=15,
    lr_layer4=0.0001,
    lr_head=0.001,
    experiment_name='resnet50_finetuned'
)
results_list.append(result4)

Experiment: resnet50_finetuned
Model: resnet50, Mode: finetuned
Trainable: 14,977,030 / 23,520,326
Epoch [1/15] Train: 0.8963, Val: 0.9387
Epoch [2/15] Train: 0.9488, Val: 0.9369
Epoch [3/15] Train: 0.9659, Val: 0.9355
Epoch [4/15] Train: 0.9793, Val: 0.9302
Epoch [5/15] Train: 0.9808, Val: 0.9320
Epoch [6/15] Train: 0.9872, Val: 0.9369
Early stopping at epoch 6


C:\Users\zhenu\AppData\Local\Temp\ipykernel_31940\2329935758.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'../checkpoints/{experime


Best Val: 0.9387, Test: 0.9290, Time: 5.90 min


In [10]:
result5 = train_model(
    model_name='efficientnet_b0',
    mode='frozen',
    num_epochs=15,
    lr=0.001,
    experiment_name='efficientnet_frozen'
)
results_list.append(result5)

Experiment: efficientnet_frozen
Model: efficientnet_b0, Mode: frozen


C:\Users\zhenu\anaconda3\envs\torch_gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\zhenu/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 29.4MB/s]


Trainable: 7,686 / 4,015,234
Epoch [1/15] Train: 0.8158, Val: 0.8913
Epoch [2/15] Train: 0.8694, Val: 0.9027
Epoch [3/15] Train: 0.8744, Val: 0.9049
Epoch [4/15] Train: 0.8775, Val: 0.9035
Epoch [5/15] Train: 0.8845, Val: 0.9020
Epoch [6/15] Train: 0.8881, Val: 0.9035
Epoch [7/15] Train: 0.8901, Val: 0.9070
Epoch [8/15] Train: 0.8880, Val: 0.9109
Epoch [9/15] Train: 0.8881, Val: 0.9074
Epoch [10/15] Train: 0.8926, Val: 0.9092
Epoch [11/15] Train: 0.8895, Val: 0.9067
Epoch [12/15] Train: 0.8908, Val: 0.9102
Epoch [13/15] Train: 0.8888, Val: 0.9116
Early stopping at epoch 13


C:\Users\zhenu\AppData\Local\Temp\ipykernel_31940\2329935758.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'../checkpoints/{experime


Best Val: 0.9116, Test: 0.8940, Time: 8.61 min


In [11]:
result6 = train_model(
    model_name='mobilenet_v2',
    mode='frozen',
    num_epochs=15,
    lr=0.001,
    experiment_name='mobilenet_frozen'
)
results_list.append(result6)

Experiment: mobilenet_frozen
Model: mobilenet_v2, Mode: frozen
Trainable: 7,686 / 2,231,558


C:\Users\zhenu\anaconda3\envs\torch_gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/15] Train: 0.8519, Val: 0.9102
Epoch [2/15] Train: 0.8898, Val: 0.9145
Epoch [3/15] Train: 0.8974, Val: 0.9195
Epoch [4/15] Train: 0.8970, Val: 0.9159
Epoch [5/15] Train: 0.8990, Val: 0.9241
Epoch [6/15] Train: 0.9055, Val: 0.9095
Epoch [7/15] Train: 0.8983, Val: 0.9177
Epoch [8/15] Train: 0.9028, Val: 0.9159
Epoch [9/15] Train: 0.9047, Val: 0.9181
Epoch [10/15] Train: 0.9004, Val: 0.9238
Early stopping at epoch 10


C:\Users\zhenu\AppData\Local\Temp\ipykernel_31940\2329935758.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'../checkpoints/{experime


Best Val: 0.9241, Test: 0.9140, Time: 6.64 min


In [12]:
comparison_df = pd.DataFrame([
    {
        'Model': r['model'],
        'Mode': r['mode'].title(),
        'LR Details': f"layer4={r.get('lr_layer4', 'N/A')}, head={r.get('lr_head', 'N/A')}"
                      if r['mode'] == 'finetuned' and 'lr_layer4' in r
                      else f"{r.get('lr', 'N/A')}",
        'Val Acc': f"{r['best_val_acc']:.4f}",
        'Test Acc': f"{r['test_acc']:.4f}",
        'Time (min)': f"{r['training_time']/60:.2f}"
    }
    for r in results_list
])

print(comparison_df.to_string(index=False))
comparison_df.to_csv('../results/transfer_learning_comparison.csv', index=False)

best = max(results_list, key=lambda x: x['test_acc'])
print(f"\n Best: {best['experiment']} - Test Acc: {best['test_acc']:.4f}")

          Model      Mode                LR Details Val Acc Test Acc Time (min)
       resnet18    Frozen                     0.001  0.9223   0.9063       9.37
       resnet18 Finetuned layer4=0.0001, head=0.001  0.9412   0.9333       4.35
       resnet50    Frozen                     0.001  0.9245   0.9120      12.84
       resnet50 Finetuned layer4=0.0001, head=0.001  0.9387   0.9290       5.90
efficientnet_b0    Frozen                     0.001  0.9116   0.8940       8.61
   mobilenet_v2    Frozen                     0.001  0.9241   0.9140       6.64

 Best: resnet18_finetuned - Test Acc: 0.9333
